In [37]:
import ee
import geemap
from datetime import datetime


In [2]:
ee.Authenticate()
ee.Initialize(project="qflfw-511214")

In [11]:
# Define variables

lon_min = -1.23
lon_max = -0.89
lat_min = 44.72
lat_max = 44.96

collection = "COPERNICUS/S2_SR_HARMONIZED"
collection_prob = "COPERNICUS/S2_CLOUD_PROBABILITY"
bbox = ee.Geometry.BBox(lon_min, lat_min, lon_max, lat_max)
bands=["B4", "B3", "B2"]
visualization = {
    'min': 0.0,
    'max': 5000,
    'bands': bands,
}
start_date = ee.Date("2022-07-01")
end_date = ee.Date("2022-07-31")
crs = "EPSG:32630"

In [12]:
def pull_dataset(start_date=start_date, end_date=end_date, bbox=bbox):
    dataset = (
        ee.ImageCollection(collection)
        .filterDate(start_date, end_date)
        .filterBounds(bbox)
    )

    return dataset

In [5]:
def check_crs(img, crs=crs, bands=bands):
    for band in bands:
        if (img.select(band).projection().getInfo()["crs"] != crs):
            raise Exception("The projection type does not match the desired one!")
        else:
            print(f"{band} OK!")

In [6]:
def display_map(dataset, bbox=bbox, vis_params=visualization, time_interval=0, zoom=9):
    m = geemap.Map()
    m.center_object(bbox, zoom=zoom)
    m.add_layer(dataset)
    if time_interval > 0:
        m.add_time_slider(dataset, vis_params=vis_params ,time_interval=time_interval)
    display(m)

In [7]:
def save_timelapse(dataset, start_date, end_date, bands=bands, dimensions=1080, output="output.gif", loop=0, frequency="day", fps=1):
    geemap.create_timelapse(
        collection=dataset,
        bands=bands,
        dimensions=dimensions,
        region=dataset.geometry(),
        start_date=start_date, 
        end_date=end_date, 
        out_gif=output,
        loop=loop,
        frequency=frequency,
        frames_per_second=fps
    )

In [13]:
# with cloud_pixel_percent = 70

dataset_jul_22 = pull_dataset(start_date, end_date)
check_crs(dataset_jul_22.first())
display_map(dataset_jul_22, time_interval=3)
save_timelapse(dataset_jul_22, start_date, end_date, output="fire_july_2022.gif")

B4 OK!
B3 OK!
B2 OK!


B4 OK!
B3 OK!
B2 OK!


Map(center=[44.83995872472239, -1.0600000000001695], controls=(WidgetControl(options=['position', 'transparent…

Total number of images: 6

ffmpeg is not installed on your computer.


Lower cloud percent allows fewer images into the collection.  
At cloud_pixel_percent = 70, we get 6 images  
At cloud_pixel_percent = 50, we get 5

In [15]:
# S2_Cloud_probability

def get_s2_sr_cld_col(start_date, end_date, aoi=bbox):
    s2_sr_col = pull_dataset(start_date, end_date, aoi)

    # Import and filter s2cloudless.
    s2_cloudless_col = (ee.ImageCollection(collection_prob)
        .filterBounds(aoi)
        .filterDate(start_date, end_date))

    # Join the filtered s2cloudless collection to the SR collection by the 'system:index' property.
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))


In [23]:
s2_sr_cld_col_eval = get_s2_sr_cld_col(bbox, start_date, end_date)

In [ ]:
def add_cloud_bands(img):
    # Get s2cloudless image, subset the probability band.
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')

    # Condition s2cloudless by the probability threshold value.
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')

    # Add the cloud probability layer and cloud mask as image bands.
    return img.addBands(ee.Image([cld_prb, is_cloud]))

def add_shadow_bands(img):
    # Identify water pixels from the SCL band.
    not_water = img.select('SCL').neq(6)

    # Identify dark NIR pixels that are not water (potential cloud shadow pixels).
    SR_BAND_SCALE = 1e4
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')

    # Determine the direction to project cloud shadow from clouds (assumes UTM projection).
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')));

    # Project shadows from clouds for the distance specified by the CLD_PRJ_DIST input.
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # Identify the intersection of dark pixels with cloud shadow projection.
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')

    # Add dark pixels, cloud projection, and identified shadows as image bands.
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

def add_cld_shdw_mask(img):
    # Add cloud component bands.
    img_cloud = add_cloud_bands(img)

    # Add cloud shadow component bands.
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # Combine cloud and shadow mask, set cloud and shadow as value 1, else 0.
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # Remove small cloud-shadow patches and dilate remaining pixels by BUFFER input.
    # 20 m scale is for speed, and assumes clouds don't require 10 m precision.
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))

    # Add the final cloud-shadow mask to the image.
    return img_cloud_shadow.addBands(is_cld_shdw)



EEException: DateRange: Cannot interpret <Geometry> as a Date.

In [ ]:
m.add_ee_layer(probability,
            {'min': 0, 'max': 100},
            'probability (cloud)', False, 1, 9)
m.add_ee_layer(clouds,
            {'palette': 'e056fd'},
            'clouds', False, 1, 9)
m.add_ee_layer(cloud_transform,
            {'min': 0, 'max': 1, 'palette': ['white', 'black']},
            'cloud_transform', False, 1, 9)
m.add_ee_layer(dark_pixels,
            {'palette': 'orange'},
            'dark_pixels', False, 1, 9)
m.add_ee_layer(shadows, {'palette': 'yellow'},
            'shadows', False, 1, 9)
m.add_ee_layer(cloudmask, {'palette': 'orange'},
            'cloudmask', True, 0.5, 9)

### Yearly composites


In [46]:
# 2020, 2021, 2022, 2023, 2024. 
# Use six bands this time: B2 (blue), B3 (green), B4 (red), B8 (NIR), B11 (SWIR1) , B12 (SWIR2)

bands_6 = ["B2", "B3", "B4", "B8", "B11", "B12"]
years = range(2020, 2025)
dataset_6 = {}

for year in years:
    dataset_6[year] = pull_dataset(start_date=ee.Date(datetime(year, 1, 1)),
                                     end_date=ee.Date(datetime(year, 12, 1, 23, 59)))
